
# Advective Cooling in One-Zone Accretion Disks

Viscously heated accretion disks do not always radiate all of their dissipated
energy locally.  When the accretion rate is high (or the disk is geometrically
thick) a substantial fraction of the viscous heating can be carried inward with
the accreting gas rather than radiated away, a process called *advective
cooling*.

:class:`~trilobite.dynamics.accretion.one_zone.core.AdvectiveDisk` extends the
standard :class:`~trilobite.dynamics.accretion.one_zone.core.FullPressureDisk` by
splitting the viscous dissipation rate into a radiated component and an
advected component:

\begin{align}q_{\rm visc} = q_{\rm rad} + q_{\rm adv}.\end{align}

The midplane temperature $T_c$ is determined at each timestep by solving
the dimensionless residual

\begin{align}f(\log T_c) = 1
        - A\,c_s^{-2}(T_c)\,T_c^4
        - B\,c_s^2(T_c)
    = 0,\end{align}

where the radiative and advective coefficients are

\begin{align}A = \frac{32}{27}\,\frac{\sigma_{\rm SB}}{\kappa(\rho,T_c)\,\alpha\,\Omega\,\Sigma^2},
    \qquad
    B = \frac{4}{3}\,\frac{\xi}{R_D^2\,\Omega^2}.\end{align}

The dimensionless entropy-gradient parameter $\xi$ sets the strength of
advective transport.  $A\,c_s^{-2}\,T_c^4$ is the fraction of viscous
heating that is radiated away, while $B\,c_s^2$ is the advected fraction.
Setting $\xi \to 0$ recovers the non-advective
:class:`~trilobite.dynamics.accretion.one_zone.core.FullPressureDisk` limit.

<div class="alert alert-info"><h4>Note</h4><p>The advective flux recorded in the output field ``Q_adv`` is computed from
    the converged closure state as

    .. math::

        \frac{q_{\rm adv}}{q_{\rm visc}}
            = \frac{4}{9\pi}\,\xi\,F_0\,\alpha\,
              \frac{M_D\,c_s^2}{R_D^4\,\Omega^2\,\Sigma},

    where $F_0 = 1 - \sqrt{R_{\rm in}/R_D}$ is the standard
    no-torque boundary factor.  This full expression is used for the
    output diagnostics; the simplified $B$ above is the form used
    in the root-finding step.</p></div>

## See Also
- :class:`~trilobite.dynamics.accretion.one_zone.core.AdvectiveDisk`
- :class:`~trilobite.dynamics.accretion.one_zone.core.FullPressureDisk`
- :meth:`~trilobite.dynamics.accretion.one_zone.base.OneZoneAccretionDiskBase.solve`

.. hint::

    For the derivation of the advective energy balance see
    `one_zone_disk_theory`.


## Setup



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy import constants as const
from astropy import units as u

from trilobite.dynamics.accretion.one_zone import FullPressureDisk, AdvectiveDisk
from trilobite.utils.plot_utils import set_plot_style

set_plot_style()

## Shared Parameters

We use a $1 \;{\rm M_{\odot}}$ BH with a $0.02 \;{\rm M_{\odot}}$ initial disk, a configuration
representative of a compact-object TDE or a neutron-star merger remnant. We can also
set the inner radius of the black hole based on the black hole's ISCO. To do this, we'll use the
:func:`~trilobite.physics_utils.general_relativity.compute_ISCO`.



In [ ]:
from trilobite.physics_utils.general_relativity import compute_ISCO
from trilobite.radiation.opacity import KramersBFESOpacity

M_BH = 1 * const.M_sun
R_in = compute_ISCO(M_BH, spin=0)
alpha = 0.1
mu = 0.62

print(R_in)

# All models share the same geometry constants so we can use any of them
# to generate consistent initial conditions.
_ref_disk = AdvectiveDisk(mu=mu, xi=1.5, opacity=KramersBFESOpacity())

ic = _ref_disk.generate_initial_conditions(
    M_BH=M_BH,
    M_D_0=0.02 * const.M_sun,
    R_D_0=5e13 * u.cm,
)

run_params = {"M_BH": M_BH, "R_in": R_in, "alpha": alpha}
t_span = (1.0e6 * u.s, 1.0e9 * u.s)
max_steps = 500_000

## Effect of the $\xi$ Parameter

We run :class:`~trilobite.dynamics.accretion.one_zone.core.AdvectiveDisk` for three values of
$\xi$ to show how the advective fraction $q_{\rm adv}/q_{\rm visc}$ scales with the
entropy-gradient parameter.



In [ ]:
xi_values = [0.1, 1.0, 1.5]
adv_results = {}

for xi_val in xi_values:
    disk = AdvectiveDisk(mu=mu, xi=xi_val, opacity=KramersBFESOpacity())
    adv_results[xi_val] = disk.solve(ic, run_params, t_span, max_steps=max_steps)

## Advective Fraction vs. Time

The advective fraction $q_{\rm adv}/q_{\rm visc}$ shows the fraction of viscous heating
that is carried inward rather than radiated.  For thin disks ($H/R \ll 1$) this
fraction is small; it grows as the disk thickens at early times.



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for xi_val, res in adv_results.items():
    t = res.data["t"].to(u.day).value
    frac = (res.data["Q_adv"] / res.data["Q_visc"]).value
    axes[0].semilogx(t, frac, label=rf"$\xi = {xi_val}$")

axes[0].set_xlabel("Time [days]")
axes[0].set_ylabel(r"$q_{\rm adv} / q_{\rm visc}$")
axes[0].set_title("Advective Fraction")
axes[0].legend()
axes[0].grid(True, which="both", ls="--", alpha=0.4)

# ---- Aspect ratio (H/R) - governs how thick the disk is ----
for xi_val, res in adv_results.items():
    t = res.data["t"].to(u.day).value
    H_over_R = res.data["H_over_R"].value
    axes[1].loglog(t, H_over_R, label=rf"$\xi = {xi_val}$")

axes[1].set_xlabel("Time [days]")
axes[1].set_ylabel(r"$H/R$")
axes[1].set_title("Disk Aspect Ratio")
axes[1].legend()
axes[1].grid(True, which="both", ls="--", alpha=0.4)

plt.tight_layout()
plt.show()

## Temperature Suppression by Advection

Because advection carries away a fraction $B\,c_s^2$ of the viscous
heating, less energy is available for radiation and the root of

\begin{align}f(\log T_c) = 1 - A\,c_s^{-2}\,T_c^4 - B\,c_s^2 = 0\end{align}

is pushed to a lower $T_c$ than the pure-radiation solution
$T_c^{(0)}$ (recovered when $\xi \to 0$).

Here we compare the midplane temperature $T_c$ for the non-advective baseline
(:class:`~trilobite.dynamics.accretion.one_zone.core.FullPressureDisk`, $\xi = 0$)
and two advective runs.



In [ ]:
disk_no_adv = FullPressureDisk(mu=mu, opacity=KramersBFESOpacity())
result_no_adv = disk_no_adv.solve(ic, run_params, t_span, max_steps=max_steps)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ---- Midplane temperature ----
t_ref = result_no_adv.data["t"].to(u.day).value
T_ref = result_no_adv.data["T_c"].to(u.K).value
axes[0].loglog(t_ref, T_ref, "k--", lw=1.5, label=r"FullPressureDisk ($\xi = 0$)")

for xi_val in [0.1, 1.0]:
    res = adv_results[xi_val]
    t = res.data["t"].to(u.day).value
    T = res.data["T_c"].to(u.K).value
    axes[0].loglog(t, T, label=rf"$\xi = {xi_val}$")

axes[0].set_xlabel("Time [days]")
axes[0].set_ylabel(r"$T_c\;[\mathrm{K}]$")
axes[0].set_title("Midplane Temperature")
axes[0].legend()
axes[0].grid(True, which="both", ls="--", alpha=0.4)

# ---- Accretion rate ----
axes[1].loglog(
    t_ref,
    result_no_adv.data["mdot"].to(u.Msun / u.yr).value,
    "k--",
    lw=1.5,
    label=r"FullPressureDisk ($\xi = 0$)",
)
for xi_val in [0.1, 1.0]:
    res = adv_results[xi_val]
    t = res.data["t"].to(u.day).value
    mdot = res.data["mdot"].to(u.Msun / u.yr).value
    axes[1].loglog(t, mdot, label=rf"$\xi = {xi_val}$")

axes[1].set_xlabel("Time [days]")
axes[1].set_ylabel(r"$\dot{M}\;[M_\odot\,\mathrm{yr}^{-1}]$")
axes[1].set_title("Accretion Rate")
axes[1].legend()
axes[1].grid(True, which="both", ls="--", alpha=0.4)

# sphinx_gallery_thumbnail_number = -1
plt.tight_layout()
plt.show()